In [1]:
import os
import gc
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False
folder_path = r'D:\clone\merged'

In [2]:
def load_and_aggregate_precip(tp_file, cp_file):
    def preprocess(ds):
        if 'valid_time' in ds.coords: ds = ds.rename({'valid_time': 'time'})
        if 'expver' in ds.dims: ds = ds.mean(dim='expver')
        return ds.sortby('time')

    ds_tp = preprocess(xr.open_dataset(os.path.join(folder_path, tp_file)))
    v_tp = list(ds_tp.data_vars)[0]
    tp_hourly = ds_tp[v_tp].mean(dim=['latitude', 'longitude']).compute()
    
    ds_cp = preprocess(xr.open_dataset(os.path.join(folder_path, cp_file)))
    v_cp = list(ds_cp.data_vars)[0]
    cp_hourly = ds_cp[v_cp].mean(dim=['latitude', 'longitude']).compute()

    # Convert kg/m2/s to mm/day (rate * 86400)
    # Daily mean rate * 86400 = total mm for the day
    df_daily = pd.DataFrame(index=tp_hourly.resample(time='1D').mean().time.values)
    df_daily['tp'] = tp_hourly.resample(time='1D').mean().values * 86400
    df_daily['cp'] = cp_hourly.resample(time='1D').mean().values * 86400
    df_daily['sp'] = df_daily['tp'] - df_daily['cp']
    df_daily['cf'] = df_daily['cp'] / (df_daily['tp'] + 1e-9)

    df_monthly = df_daily.resample('ME').mean()
    df_yearly_sum = df_daily.resample('YS').sum()
    df_yearly_mean = df_daily.resample('YS').mean()
    
    climatology = df_monthly.groupby(df_monthly.index.month).mean()
    df_monthly['month'] = df_monthly.index.month
    df_monthly['year'] = df_monthly.index.year
    df_monthly['tp_anom'] = df_monthly['tp'] - df_monthly['month'].map(climatology['tp'])

    ds_tp.close(); ds_cp.close(); gc.collect()
    return df_daily, df_monthly, df_yearly_sum, df_yearly_mean, climatology

In [3]:
tp_file = 'mean_total_precipitation_rate_merged.nc'
cp_file = 'mean_convective_precipitation_rate_merged.nc'

df_daily, df_monthly, df_yearly_sum, df_yearly_mean, clim = load_and_aggregate_precip(tp_file, cp_file)
df_daily.head()

In [4]:
plt.figure(figsize=(10, 6))
plt.plot(clim.index, clim['tp'], color='black', lw=2.5, label='Tổng lượng mưa (Total)')
plt.plot(clim.index, clim['sp'], color='#85B7EB', lw=2, ls='--', label='Mưa tầng (Stratiform)')
plt.plot(clim.index, clim['cp'], color='#D85A30', lw=2, ls=':', label='Mưa đối lưu (Convective)')
plt.title("1. Khí hậu học lượng mưa theo mùa")
plt.xlabel("Tháng")
plt.ylabel("Lượng mưa trung bình (mm/ngày)")
plt.xticks(range(1, 13))
plt.legend()
plt.show()

In [5]:
plt.figure(figsize=(12, 6))
x_years = df_yearly_sum.index.year
y_total = df_yearly_sum['tp']

plt.bar(x_years, y_total, color='lightgreen', alpha=0.5, label='Lượng mưa năm')
plt.plot(x_years, y_total.rolling(5, center=True).mean(), color='green', lw=2, label='Trung bình trượt 5 năm')

slope, intercept, r, p, _ = stats.linregress(x_years, y_total)
plt.plot(x_years, intercept + slope * x_years, color='red', ls='--', label=f"Xu hướng: {slope*10:.1f} mm/thập kỷ")

plt.title("2. Biến động lượng mưa tổng cộng hàng năm")
plt.xlabel("Năm")
plt.ylabel("Tổng lượng mưa (mm/năm)")
plt.legend()
plt.show()

In [6]:
plt.figure(figsize=(12, 6))
x_years = df_yearly_mean.index.year
y_cf = df_yearly_mean['cf']

slope, intercept, r, p, _ = stats.linregress(x_years, y_cf)
sns.regplot(x=x_years, y=y_cf, scatter_kws={'alpha':0.5}, line_kws={'color':'red', 'ls':'--'})

plt.title(f"3. Xu hướng tỷ lệ mưa đối lưu (Trend: {slope*10:.4f}/decade, p={p:.4f})")
plt.xlabel("Năm")
plt.ylabel("Tỷ lệ mưa đối lưu (0-1)")
plt.show()

In [7]:
pivot = df_monthly.pivot(index='year', columns='month', values='tp_anom')
plt.figure(figsize=(15, 8))
sns.heatmap(pivot, cmap="BrBG", center=0, cbar_kws={'label': 'Dị thường (mm/ngày)'})
plt.title("4. Ma trận dị thường lượng mưa hàng tháng")
plt.show()

In [8]:
plt.figure(figsize=(16, 5))
plt.stackplot(df_monthly.index, 
              df_monthly['cp'], df_monthly['sp'], 
              labels=['Mưa đối lưu', 'Mưa tầng'], 
              colors=['#D85A30', '#85B7EB'])
plt.title("5. Phân hóa tính chất mưa: Đối lưu vs Tầng")
plt.ylabel("mm/ngày")
plt.legend(loc='upper left')
plt.show()

In [9]:
thresholds = {"Nhẹ (>1)": 1, "Vừa (>10)": 10, "To (>25)": 25, "Rất to (>50)": 50}
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
axes = axes.flatten()

for i, (label, thresh) in enumerate(thresholds.items()):
    counts = (df_daily['tp'] > thresh).resample('YS').sum()
    axes[i].bar(counts.index.year, counts.values, color='steelblue')
    axes[i].set_title(f"Số ngày mưa {label}")
    axes[i].set_ylabel("Số ngày")

plt.suptitle("6. Tần suất các ngày mưa cực đoan theo ngưỡng")
plt.tight_layout()
plt.show()

In [10]:
df_daily['decade'] = (df_daily.index.year // 10) * 10
plt.figure(figsize=(10, 6))

for decade in sorted(df_daily['decade'].unique()):
    subset = df_daily[df_daily['decade'] == decade]['tp']
    vals = subset[subset > 1].values
    sns.ecdfplot(vals, label=f"{decade}s")

plt.xscale('log')
plt.title("7. Sự dịch chuyển phân phối lượng mưa theo thập kỷ (Hàm phân phối tích lũy)")
plt.xlabel("Lượng mưa (mm/ngày) - Thang Log")
plt.ylabel("Xác suất tích lũy")
plt.legend()
plt.show()

In [11]:
def get_max_dry_spell(series):
    is_dry = series < 1.0
    return (is_dry != is_dry.shift()).cumsum().where(is_dry).value_counts().max()

annual_dry = df_daily['tp'].groupby(pd.Grouper(freq='YS')).apply(get_max_dry_spell)

plt.figure(figsize=(12, 6))
plt.plot(annual_dry.index.year, annual_dry.values, marker='o', color='brown')
plt.title("8. Độ dài đợt khô hạn nhất trong năm (Dry Spell Length)")
plt.xlabel("Năm")
plt.ylabel("Số ngày liên tiếp mưa < 1mm")
plt.show()

In [12]:
df_daily[['tp', 'cp', 'sp', 'cf']].to_csv(os.path.join(folder_path, 'derived_precip_daily.csv'))